In [1]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
import utils_2Q_gate_zp as ut
ut.set_fig_font() ### Set various sizes in plotting
import scipy as sp
from joblib import Parallel, delayed
import itertools
from qutip.qip.operations import rz, cz_gate, cnot, rx, hadamard_transform, swap
import pandas as pd

from pathlib import Path
import ham_data as hd

# With npz

In [2]:
folder_load = '../../data/_truc_3000'
data = np.load(Path(folder_load, 'two_qubit_data.npz'), allow_pickle=True)
print(list(data.keys()))

['eval1', 'eval2', 'evecs1', 'evecs2', 'n_theta1', 'n_theta2', 'hspace_1_charge', 'hspace_2_charge', 'g_theta1theta2', 'evals_tot', 'evecs_tot', 'hspace_full', 'n_theta1_dressed', 'n_theta2_dressed', 'top_idx', 'top_overlap', 'hspace_n_theta1', 'hspace_n_theta2', 'params']


In [6]:
# print(data['top_idx'])
print(data['hspace_full'])
# print(data['eval1'])
# print(data['evals_tot'])

['0-0' '0-1' '1-0' ... '57-14' '56-17' '56-16']


In [5]:
[hspace_full, eket_tot, eval_tot, n_theta0_dress, 
    n_theta1_dress, hspace_0, hspace_1, logi_state] = hd.load_two_qubit_data(folder_load, return_full=False)

### Get fidelity for input params (pick=True)

In [8]:
truc_full = 1000
truc_list = np.arange(truc_full)
hspace_full = hspace_full[:truc_full]
eval_tot = eval_tot[:truc_full]
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)
W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]

num_cpus = 16
x0_vec = np.array([
 [182.99766, 0.007776, 0.01307, -4.42509061, -4.37463037],
])
n_job = 100
c_op_list = []
H0_full = qt.Qobj(np.diag(eval_tot))
logi_idx_full = [hspace_full.index(i) for i in logi_state]
H_drive_full = [H0_full, [n_theta1_dress, ut.drive_gauss_A] ]

In [9]:
arg_full = [H_drive_full, W_20_50, num_cpus, c_op_list, logi_idx_full ]
f_full = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_full)
                                            for args_indep in x0_vec[:, :3])
print(f' fidelity (dim={truc_full}) =', np.round(f_full, 8).tolist())

ValueError: not enough values to unpack (expected 7, got 5)

In [ ]:
# len_part = 190
# hspace_part = hspace_full[:len_part]
# index_part = np.arange(len_part)
# H0_part = ut.truncate_2( H0_full, index_part )
# n_theta1_part = ut.truncate_2(n_theta1_dress, index_part)
# logi_idx_part = [hspace_part.index(i) for i in logi_state]
# H_drive_part = [H0_part, [n_theta1_part, ut.drive_gauss_A] ]

# arg_part = [H_drive_part, W_20_50, num_cpus, c_op_list, logi_idx_part ]
# f_part = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_part)
#                                             for args_indep in x0_vec[:, :3])
# print(f' fidelity (dim={len_part},{charge_pick}) =', np.round(f_part, 8).tolist())

 fidelity (dim=190,True) = [-0.29474067, -2.21903731, -3.9384408]


# Without npz

In [2]:
truc1, truc_tot, charge_pick = 300, 2000, False
truc_full = 1000

folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')
overlap =  np.load(folder+f'top_overlap_300_{truc_tot}_{charge_pick}.npy')
index =  np.load(folder+f'top_index_300_{truc_tot}_{charge_pick}.npy')
# overlap = np.load(folder+'top_overlap.txt')
# index = np.load(folder+'top_index.txt')

In [3]:
print(np.shape(overlap), np.shape(index))
print(overlap[0], index[0])

(2000, 10) (2000, 10, 2)
[9.99939628e-01 1.09788403e-02 2.08593424e-04 1.85955180e-04
 1.79130044e-04 1.68434043e-04 1.41582363e-04 1.27356954e-04
 1.21454033e-04 6.13140683e-05] [[ 0  0]
 [ 1  1]
 [ 1  8]
 [ 8  1]
 [ 0  4]
 [ 4  0]
 [ 1 26]
 [26  1]
 [ 4  4]
 [ 1 33]]


In [24]:
state = '0-8'
idx = hspace_full.index(state)
print(idx)
superposition = []
for i in range(4):
    # print(np.round(overlap[idx, i]**2, 3), index[idx, i])
    superposition.append( str(np.round(overlap[idx, i]**2, 3))+ str(index[idx, i]) )
print(f'{idx}: {state}=', superposition)

20
20: 0-8= ['0.963[0 8]', '0.035[1 4]', '0.001[4 1]', '0.0[ 1 12]']


In [34]:
for idx in np.arange(400,600):
    if overlap[idx, 0]**2 < 0.6:
        superposition = []
        for i in range(4):
            # print(np.round(overlap[idx, i]**2, 3), index[idx, i])
            superposition.append( str(np.round(overlap[idx, i]**2, 3))+ str(index[idx, i]) )
        print(f'{idx}: {hspace_full[idx]}=', superposition)

408: 12-9= ['0.561[12  9]', '0.247[18  5]', '0.095[15  8]', '0.031[ 9 12]']
415: 18-6= ['0.449[18  6]', '0.372[12 10]', '0.116[ 5 19]', '0.032[ 8 17]']
425: 12-10= ['0.481[18  6]', '0.454[12 10]', '0.023[ 8 17]', '0.01[13 10]']
459: 36-1= ['0.579[36  1]', '0.368[49  0]', '0.016[27  4]', '0.011[14  9]']
526: 11-17= ['0.552[11 17]', '0.229[17 11]', '0.105[16 10]', '0.098[ 7 23]']
527: 8-18= ['0.565[ 8 18]', '0.333[12 12]', '0.034[18  8]', '0.009[ 4 30]']
528: 16-11= ['0.6[16 11]', '0.201[17 10]', '0.089[11 15]', '0.046[21  7]']
534: 12-12= ['0.424[12 12]', '0.37[ 8 18]', '0.146[18  8]', '0.008[ 8 21]']
558: 28-4= ['0.511[28  4]', '0.259[54  0]', '0.202[39  1]', '0.022[41  1]']
583: 12-16= ['0.432[12 16]', '0.362[15 12]', '0.094[ 8 20]', '0.047[22  8]']
586: 20-8= ['0.471[20  8]', '0.346[12 13]', '0.099[ 8 21]', '0.034[13 12]']
587: 4-30= ['0.406[ 4 30]', '0.281[20  8]', '0.205[12 13]', '0.052[ 8 21]']
588: 12-13= ['0.519[ 4 30]', '0.281[12 13]', '0.127[20  8]', '0.023[ 8 18]']


### Get fidelity for input params (pick=True)

In [ ]:
truc1, truc_tot, charge_pick = 300, 1000, True
truc_full = 1000

folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')

truc_list = np.arange(truc_full)
hspace_full = hspace_full[:truc_full]
eval_tot = eval_tot[:truc_full]
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)
logi_state = ['0-0', '0-2', '2-0', '2-2']
W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]

num_cpus = 16
x0_vec = np.array([
 [182.99766, 0.007776, 0.01307, -4.42509061, -4.37463037],
])
n_job = 100
c_op_list = []
H0_full = qt.Qobj(np.diag(eval_tot))
logi_idx_full = [hspace_full.index(i) for i in logi_state]
H_drive_full = [H0_full, [n_theta1_dress, ut.drive_gauss_A] ]

In [7]:
arg_full = [H_drive_full, W_20_50, num_cpus, c_op_list, logi_idx_full ]
f_full = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_full)
                                            for args_indep in x0_vec[:, :3])
print(f' fidelity (dim={truc_full},{charge_pick}) =', np.round(f_full, 8).tolist())

 fidelity (dim=1000,True) = [-4.2781217]


In [ ]:
# len_part = 190
# hspace_part = hspace_full[:len_part]
# index_part = np.arange(len_part)
# H0_part = ut.truncate_2( H0_full, index_part )
# n_theta1_part = ut.truncate_2(n_theta1_dress, index_part)
# logi_idx_part = [hspace_part.index(i) for i in logi_state]
# H_drive_part = [H0_part, [n_theta1_part, ut.drive_gauss_A] ]

# arg_part = [H_drive_part, W_20_50, num_cpus, c_op_list, logi_idx_part ]
# f_part = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_part)
#                                             for args_indep in x0_vec[:, :3])
# print(f' fidelity (dim={len_part},{charge_pick}) =', np.round(f_part, 8).tolist())

 fidelity (dim=190,True) = [-0.29474067, -2.21903731, -3.9384408]


### Get fidelity for input params (pick=False)

In [3]:
truc1, truc_tot, charge_pick = 300, 2000, False
truc_full = 2000
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')

truc_list = np.arange(truc_full)
hspace_full = hspace_full[:truc_full]
eval_tot = eval_tot[:truc_full]
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)
W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]

H0_full = qt.Qobj(np.diag(eval_tot))
logi_idx_full = [hspace_full.index(i) for i in logi_state]
H_drive_full = [H0_full, [n_theta1_dress, ut.drive_gauss_A] ]

In [ ]:
arg_full = [H_drive_full, W_20_50, num_cpus, c_op_list, logi_idx_full ]
f_full = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_full)
                                            for args_indep in x0_vec[:, :3])
print(f' fidelity (dim={truc_full},{charge_pick}) =', np.round(f_full, 8).tolist())

 fidelity (dim=2000,False) = [-4.14998241]


: 

In [ ]:
# len_part = 500
# hspace_part = hspace_full[:len_part]
# index_part = np.arange(len_part)
# H0_part = ut.truncate_2( H0_full, index_part )
# n_theta1_part = ut.truncate_2(n_theta1_dress, index_part)
# logi_idx_part = [hspace_part.index(i) for i in logi_state]
# H_drive_part = [H0_part, [n_theta1_part, ut.drive_gauss_A] ]

# arg_part = [H_drive_part, W_20_50, num_cpus, c_op_list, logi_idx_part ]
# f_part = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_part)
#                                             for args_indep in x0_vec)
# print(f' fidelity (dim={truc_full},{charge_pick}) =', np.round(f_full, 8).tolist())
# print(f' fidelity (dim={len_part},{charge_pick}) =', np.round(f_part, 8).tolist())

 fidelity (dim=500,False) = [-0.2947592, -2.2188636, -3.94444026]


In [ ]:
if len(c_op_list) == 0 and H0.isoper:
    # calculate propagator for the wave function

    N = H0.shape[0]
    dims = H0.dims

    if parallel:
        unitary_mode = 'single'
        u = np.zeros([N, N, len(tlist)], dtype=complex)
        output = parallel_map(_parallel_sesolve, range(N),
                                task_args=(N, H, tlist, args, options),
                                progress_bar=progress_bar, num_cpus=num_cpus)
        for n in range(N):
            for k, t in enumerate(tlist):
                u[:, n, k] = output[n].states[k].full().T    
else:
    # calculate the propagator for the vector representation of the
    # density matrix (a superoperator propagator)
    unitary_mode = 'single'
    N = H0.shape[0]
    dims = [H0.dims, H0.dims]

    u = np.zeros([N * N, N * N, len(tlist)], dtype=complex)

    if parallel:
        output = parallel_map(_parallel_mesolve, range(N * N),
                                task_args=(
                                    N, H, tlist, c_op_list, args, options),
                                task_kwargs={"dims": H0.dims},
                                progress_bar=progress_bar, num_cpus=num_cpus)
        for n in range(N * N):
            for k, t in enumerate(tlist):
                u[:, n, k] = mat2vec(output[n].states[k].full()).T

def _parallel_sesolve(n, N, H, tlist, args, options):
    psi0 = basis(N, n)
    output = sesolve(H, psi0, tlist, [], args, options, _safe_mode=False)
    return output                